# 2 — Covalently modify a protein and simulate it

The story in four moves:

1. **Here is a protein I want** — loaded with full chemistry (notebook 1).
2. **Here is an mBuild Compound I built from scratch** — a fragment written
   as star-sited SMILES; the `*` marks where the bond forms.
3. **Attach them** — one call places, bonds, clash-checks, and relaxes.
4. **The output is exactly what OpenFF needs** — the prepared PDB + one bond
   record feed Pablo, NAGL assigns charges, Interchange assigns ff14SB +
   Sage parameters, and OpenMM runs solvated MD.

(Run notebook 1 first so `1ubq_protonated.pdb` exists.)


In [11]:
from mbuild.biopolymers import Protein, prepare_fragment

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues | net formal charge:",
      protein.net_formal_charge)


76 residues | net formal charge: 0


## 1. Prepare your fragment

Write your fragment as SMILES with a `*` marking where the bond forms
(the star becomes the leaving hydrogen). Charges in the SMILES are kept.
Here: an octanoyl group, starred at the carbonyl carbon.


In [12]:
#fragment = prepare_fragment("*C(=O)CCCCCCC", "OCT")
fragment = prepare_fragment("CC1(C2=C(C=CC(=C2)S(=O)(=O)O)[N+](=C1C=CC=CC=C3C(C4=C(N3CCCS(=O)(=O)[O-])C=CC(=C4)S(=O)(=O)O)(C)CCCCC(=O)NCC*)CCCS(=O)(=O)O)C", "FRE")
print("bond site:", fragment.link_atoms)


INFO:mbuild.biopolymers.fragments:Fragment 'Compound' wrapped into residue 'FRE'.
INFO:mbuild.biopolymers.fragments:Renamed atoms of residue FRE to element+index names so they are unique within the residue.


bond site: {'1': 'C33'}


## 2. Attach it

Pick the protein site by residue number + atom name. The fragment already
knows its own bond site from the `*`. One hydrogen leaves each side; the
fragment is aligned and bonded. If it lands too close to the protein,
mBuild relaxes it automatically with the protein held fixed (generic
parameters, protein coordinates unchanged; opt out with `relax=False`).


In [13]:
record = protein.attach(fragment, resnum=63, atom_name="NZ", chain_id="A")
print("new bond:", record.residue1.name, record.atom1_name, "-",
      record.residue2.name, record.atom2_name)
print("leaving hydrogens:", record.leaving1, record.leaving2)


new bond: LYS NZ - FRE C33
leaving hydrogens: ('HZ1',) ('H1',)


## 3. Write the modified PDB + bond records

`save_pdb` writes a standards-conformant file (residues, chains, TER, CONECT).
`bond_records()` returns one neutral record per new bond — everything a
downstream loader needs to know about the modification.


In [14]:
#protein.save_pdb("1ubq_octanoyl.pdb", overwrite=True)
protein.save_pdb("1ubq_FRET.pdb", overwrite=True)
records = protein.bond_records()
records


[{'residue_names': ('LYS', 'FRE'),
  'residue_numbers': (63, 77),
  'atom_names': ('NZ', 'C33'),
  'leaving_atoms': (['HZ1'], ['H1']),
  'bond_order': 1}]

## 4a. Ingest with OpenFF (needs `openff-pablo >= 0.2` + `openff-toolkit`)

The fragment is not a CCD residue, so give Pablo one named definition built
from the same SMILES, then pass the spec straight through.


In [5]:
from rdkit import Chem
from openff.toolkit import Molecule
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb

# Build the definition from the SAME starred SMILES: replace the star
# with H exactly as prepare_fragment did, so atom order matches.
star = Chem.RWMol(Chem.MolFromSmiles("*C(=O)CCCCCCC"))
for atom in star.GetAtoms():
    if atom.GetAtomicNum() == 0:
        atom.SetAtomicNum(1)
mol = star.GetMol()
Chem.SanitizeMol(mol)
offmol = Molecule.from_rdkit(Chem.AddHs(mol), allow_undefined_stereo=True)
for atom, particle in zip(offmol.atoms, fragment.particles()):
    atom.name = particle.name

# Format the neutral record into pablo's with_crosslink vocabulary.
record = records[0]
spec = {
    "residues": list(record["residue_names"]),
    "linking_atoms": list(record["atom_names"]),
    "leaving_atoms": [list(side) for side in record["leaving_atoms"]],
    "bond_order": record["bond_order"],
}
library = STD_CCD_CACHE.with_(
    {"OCT": [ResidueDefinition.from_molecule(offmol, residue_name="OCT")]}
).with_crosslink(**spec)

topology = topology_from_pdb("1ubq_octanoyl.pdb", residue_library=library)
molecule = topology.molecule(0)
print(molecule.n_atoms, "atoms | net charge:", molecule.total_charge)


1254 atoms | net charge: 0.0 elementary_charge


## 5. Assign force-field parameters

MosDef has no biopolymer force fields yet, so this is where the OpenFF
ecosystem takes over — using nothing but the objects above: NAGL graph
charges for the whole modified protein in one call, then the Amber ff14SB
port + Sage through Interchange.


In [6]:
from openff.toolkit import ForceField
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

conjugate = molecule  # the modified protein from Pablo, above
conjugate.assign_partial_charges("openff-gnn-am1bcc-0.1.0-rc.3.pt",
                                 toolkit_registry=NAGLToolkitWrapper())
print("NAGL net charge:", conjugate.partial_charges.sum())

force_field = ForceField("ff14sb_off_impropers_0.0.4.offxml",
                         "openff-2.2.1.offxml")


NAGL net charge: 2.6645352591003757e-14 elementary_charge


## 6. Solvate and run MD with OpenMM


In [7]:
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.toolkit import Molecule, Topology
from openff.units import unit as off_unit

water = Molecule.from_smiles("O")
water.generate_conformers(n_conformers=1)
for atom in water.atoms:
    atom.metadata["residue_name"] = "HOH"

solvated = pack_box([water], [1500],
                    solute=Topology.from_molecules([conjugate]),
                    target_density=0.95 * off_unit.gram / off_unit.milliliter,
                    box_shape=UNIT_CUBE,
                    tolerance=2.0 * off_unit.angstrom)
print("solvated:", solvated.n_atoms, "atoms")

interchange = force_field.create_interchange(
    solvated, charge_from_molecules=[conjugate])


solvated: 5754 atoms


In [ ]:
import openmm
from openmm import unit

simulation = interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picosecond, 2.0 * unit.femtosecond),
)
simulation.minimizeEnergy(maxIterations=200)
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
for block in range(2):
    simulation.step(250)
    state = simulation.context.getState(getEnergy=True)
    print(f"step {(block + 1) * 250}: PE =", state.getPotentialEnergy())
print("MD OK — a covalently modified protein, built in mBuild, running in OpenMM.")


## Recap

- **mBuild** owned the coordinates: strict protein loading, fragment
  definition, covalent attachment with sensible geometry, chemistry-complete
  exports.
- **OpenFF** owned the parameters: Pablo ingestion, NAGL charges,
  ff14SB + Sage through Interchange.
- More: multi-site tethering, glycans, GLYCAM fragments, mixed force fields,
  and a PyMOL placement movie live on this repository's `showcase-extras`
  branch.
